In [35]:
#linking custom qda model / recreating specific splits and variables to evaluate the math performance
import sys
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from scipy.stats import multivariate_normal

# Find project root
project_root = Path.cwd().resolve()
while not (project_root / 'app').exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.services.custom_qda import CustomQDA

# Load model
custom_qda_model = joblib.load('custom_qda_model_for_flask.pkl')

# Load data
df_abt = pd.read_csv('../../data/processed/aml_clean.csv')
df_abt['date'] = pd.to_datetime(df_abt['date']).dt.date
df_abt['time'] = pd.to_datetime(df_abt['time'], format='%H:%M:%S').dt.time
df_abt_cleaned = df_abt.dropna(subset=['is_laundering'])

# Variables
key_vars = ['date', 'time', 's_bank_name', 's_bank_id', 's_entity_id',
       's_entity_name', 's_entity_type','s_latitude',
       's_longitude', 'r_bank_name', 'r_bank_id', 'r_entity_id',
       'r_entity_name', 'r_entity_type', 'r_latitude', 'r_longitude']
target = "is_laundering"

# Recreate splits
df_abt_subset, _ = train_test_split(df_abt_cleaned, train_size=0.1, random_state=42, stratify=df_abt_cleaned['is_laundering'])
train_df, test_df = train_test_split(df_abt_subset, test_size=0.2, random_state=42, stratify=df_abt_subset['is_laundering'])

X_train_raw = train_df.drop(columns=key_vars + [target])
y_train = train_df[target]
X_test_raw = test_df.drop(columns=key_vars + [target])
y_test = test_df[target]

# Load pipeline and transform
loaded_preprocessing_pipeline = joblib.load('../../analysis/features/pycaret_preprocessing_pipeline.pkl')
X_train_preprocessed = loaded_preprocessing_pipeline.transform(X_train_raw)
X_test_preprocessed = loaded_preprocessing_pipeline.transform(X_test_raw)

# Recreate SMOTE
smote_instance = SMOTE(k_neighbors=3, random_state=42, sampling_strategy=0.2)
X_train_smoted, y_train_smoted = smote_instance.fit_resample(X_train_preprocessed, y_train)

print("Everything loaded successfully!")

Everything loaded successfully!


In [23]:
# Verifying priors: 
# Do the priors reflect the actual class distribution in training data?
# They should match ratio of laundering v non-laundering after SMOTE
print("Class priors:")

for c in custom_qda_model.classes:
    print(f" Class {c}: {custom_qda_model.priors[c]:.4f}")

Class priors:
 Class 0: 0.8333
 Class 1: 0.1667


In [24]:
# Verifying math means
# Are the means reasonable? and do they differ between classes?

print("Mean vector shape: ", custom_qda_model.means[0].shape)
print("Mean difference (class 1 - class 0): ")
print(custom_qda_model.means[1] - custom_qda_model.means[0])

Mean vector shape:  (56,)
Mean difference (class 1 - class 0): 
from_bank                               1.244641e-04
account                                 2.160251e-03
to_bank                                 3.013735e-04
account.1                               1.815373e-03
amount_received                         9.395088e-07
receiving_currency_US Dollar           -1.030612e-02
receiving_currency_Rupee               -5.387724e-03
receiving_currency_Euro                 6.822953e-02
receiving_currency_UK Pound            -3.449562e-03
receiving_currency_Canadian Dollar     -4.279645e-03
receiving_currency_Bitcoin             -2.775262e-02
receiving_currency_Yuan                -1.308319e-02
receiving_currency_Shekel              -2.140990e-02
receiving_currency_Swiss Franc         -2.153885e-02
receiving_currency_Australian Dollar   -7.751332e-03
receiving_currency_Saudi Riyal          5.723486e-02
receiving_currency_Mexican Peso        -9.608314e-03
receiving_currency_Yen             

In [25]:
# Verifying covariance Matricies 
# Are diagonals zero / close to zero? 
# Are classes covariance similar or different? 

print("Covariance matrix shape: ", custom_qda_model.covs[0].shape)
print("Class 0 covariance diagonal: ", np.diag(custom_qda_model.covs[0])[:5])
print("Class 1 covriance diagonal: ", np.diag(custom_qda_model.covs[1])[:5]) 


Covariance matrix shape:  (56, 56)
Class 0 covariance diagonal:  [0.14000364 0.14000182 0.14000535 0.14000349 0.14      ]
Class 1 covriance diagonal:  [0.14000553 0.14014747 0.14000864 0.14017909 0.14      ]


In [ ]:
# manually verifying the regularization formula
raw_cov = np.cov(X_train_smoted[y_train_smoted == 1], rowvar=False)
identity = np.eye(raw_cov.shape[0])
expected_cov = (1 - 0.14) * raw_cov + 0.14 * identity
actual_cov = custom_qda_model.covs[1]

print("Regularization check (should be near zero):")
print(np.max(np.abs(expected_cov - actual_cov)))

Regularization check (should be near zero):
0.0


In [32]:
# verifying probabilities sum to 1 

probability_matrix = custom_qda_model.predict_proba(X_test_preprocessed)
row_sums = np.sum(probability_matrix, axis=1)

print("Min row sum: ", row_sums.min())
print("Max row sum: ", row_sums.max())
print("Any rows not summing to 1: ", not np.allclose(row_sums, 1.0))
      #np.any(np.abs(row_sums - 1) > 1e-6))

Min row sum:  0.9999999999999998
Max row sum:  1.0000000000000002
Any rows not summing to 1:  False


In [ ]:
# verifying bayes theorem is applied correctly via a single sample 
# Outputs for manual and model should be identical 
#sampling 
from pickle import TRUE


X_test_array = np.array(X_test_preprocessed)
sample = X_test_array[0].reshape(1, -1)
print("Sample shape: ", sample.shape)

#manual compute for each class
manual_probabilities = []
for i in custom_qda_model.classes:
    likelihood = multivariate_normal.pdf(sample, mean=custom_qda_model.means[i], cov=custom_qda_model.covs[i], allow_singular = TRUE)
    posterior = likelihood * custom_qda_model.priors[i]
    manual_probabilities.append(posterior)

manual_probabilities = np.array(manual_probabilities)
manual_probabilities /= manual_probabilities.sum() 

print("Manual probabilities: ", manual_probabilities)
print("Model probabilities: ", custom_qda_model.predict_proba(sample))

Sample shape:  (1, 56)
Manual probabilities:  [0.71159555 0.28840445]
Model probabilities:  [[0.71159555 0.28840445]]


In [ ]:
# checking for singular matrix risk
# any eigenvalue at or near zero means that the covariance matrix is signular which could 
# caus instibility in the model

for i in custom_qda_model.classes:
    cov_matrix = custom_qda_model.covs[i]
    eigenvalues = np.linalg.eigvals(cov_matrix)
    print(f"Class {i} covariance matrix eigenvalues: {eigenvalues[:5]}")

Class 0 covariance matrix eigenvalues: [0.7627977  0.41105186 0.39961926 0.33083403 0.2894721 ]
Class 1 covariance matrix eigenvalues: [0.76656377 0.47690793 0.38253691 0.2490426  0.23709059]
